In [1]:
#############################################
# Notebook 05 – Hedonisches Regressionsmodell
# Projekt: Preisbewertung NL
#############################################

import pandas as pd
import numpy as np

# Saubere Basis laden
df = pd.read_csv("../data_clean/df_model_prepared.csv")

# Features definieren (Beispiel – du setzt hier deine echten Features ein)
feature_cols = [
    "population",
    "income_mean",
    "living_space_total",
    "ratio_price_model",
    "ratio_area"
]

X = df[feature_cols].copy()

# Target definieren
y = df["log_price_total"].copy()

# Exportieren
X.to_csv("../data_clean/X_features.csv", index=False)
y.to_csv("../data_clean/y_target.csv", index=False)

print("X:", X.shape)
print("y:", y.shape)


X: (341, 5)
y: (341,)


In [2]:
####################################################
# 5.1. Vorbereitung eines weiteren Regressionsmodell
####################################################

import pandas as pd
import numpy as np
import statsmodels.api as sm

# Schritt 1: WOZ‑Grundlage einlesen

# WOZ-Basis laden
df_woz = pd.read_csv("../data_clean/df_master_woz_2023.csv")

print("WOZ-Basis:", df_woz.shape)
df_woz.head()


WOZ-Basis: (341, 2)


,municipality_code,woz_mean
0,GM0014,294000.0
1,GM0034,363000.0
2,GM0037,242000.0
3,GM0047,212000.0
4,GM0050,393000.0


In [3]:
# Schritt 2: df_model_prepared laden und mit WOZ mergen

df = pd.read_csv("../data_clean/df_model_prepared.csv")
df_woz = pd.read_csv("../data_clean/df_master_woz_2023.csv")

# Merge mit Suffixen, damit wir sehen, was passiert
df_ext = df.merge(df_woz, on="municipality_code", how="left", suffixes=("", "_woz"))

# Falls df_model_prepared bereits eine woz_mean hatte → überschreiben
if "woz_mean_woz" in df_ext.columns:
    df_ext["woz_mean"] = df_ext["woz_mean_woz"]
    df_ext = df_ext.drop(columns=["woz_mean_woz"])


In [4]:
# Schritt 3: Logarithmen berechnen

df_ext["log_price_total"] = np.log(df_ext["price_total"])
df_ext["log_woz_mean"] = np.log(df_ext["woz_mean"])


In [5]:
# Schritt 4: residual_woz berechnen

X = sm.add_constant(df_ext["log_woz_mean"])
y = df_ext["log_price_total"]

model_woz = sm.OLS(y, X).fit()
df_ext["residual_woz"] = model_woz.resid

print(model_woz.summary())


                            OLS Regression Results                            
Dep. Variable:        log_price_total   R-squared:                       0.920
Model:                            OLS   Adj. R-squared:                  0.920
Method:                 Least Squares   F-statistic:                     3917.
Date:                Sat, 21 Mar 2026   Prob (F-statistic):          2.55e-188
Time:                        14:38:16   Log-Likelihood:                 474.60
No. Observations:                 341   AIC:                            -945.2
Df Residuals:                     339   BIC:                            -937.5
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            1.3602      0.185      7.364   

In [6]:
# Schritt 5: Erweiterte Datei speichern

cols_residual = [
    "municipality_code",
    "gemeentenaam",
    "woz_mean",
    "log_woz_mean",
    "price_total",
    "log_price_total",
    "residual_woz",
    "is_randstad",
    "income_mean",
    "bevolkingsdichtheid",
    "share_egw",
    "avg_space_egw",
    "migration_rate",
    "woningvoorraad_total",
    "pipeline_total",
    "employment_rate",
    "ratio_price_model"
]

df_residual = df_ext[cols_residual].copy()

df_residual.to_csv("../data_clean/df_residual_2023.csv", index=False)

print("df_residual:", df_residual.shape)


df_residual: (341, 17)
